In [1]:
import torch

print(torch.__version__)

2.14.0+cu130


Sample code example for creating a NN from using pytorch.

In [2]:
import torch 
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [3]:
# download the training data from torchvision datasets
train_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_date = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)


100%|██████████| 26.4M/26.4M [00:13<00:00, 1.92MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 225kB/s]
100%|██████████| 4.42M/4.42M [00:03<00:00, 1.27MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 9.93MB/s]


In [4]:
batch_size = 64

# create data loaders 
train_dataloder = DataLoader(train_data, batch_size=batch_size)
test_dataloder = DataLoader(test_date, batch_size=batch_size)

for x, y in train_dataloder:
    print(f"Shape of x [N, C, H, W]: {x.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of x [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [5]:
# create the Nural Network Model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [6]:
# define the model 

class ImageClassificationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = ImageClassificationModel().to(device)
print(model)

ImageClassificationModel(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [8]:
input_image = torch.rand(3, 28, 28, device=device)
print(input_image.size())

# flattern the input image to a 1D tensor
flattern = nn.Flatten()
flatterned_image = flattern(input_image)
print(flatterned_image.size())



torch.Size([3, 28, 28])
torch.Size([3, 784])


In [9]:
# train the model
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)


In [10]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [12]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloder, model, loss_fn, optimizer)

Epoch 1
-------------------------------
loss: 1.145571  [    0/60000]
loss: 1.141995  [ 6400/60000]
loss: 0.959212  [12800/60000]
loss: 1.095280  [19200/60000]
loss: 0.972218  [25600/60000]
loss: 1.006685  [32000/60000]
loss: 1.046237  [38400/60000]
loss: 0.982306  [44800/60000]
loss: 1.019439  [51200/60000]
loss: 0.957607  [57600/60000]
Epoch 2
-------------------------------
loss: 1.036658  [    0/60000]
loss: 1.053532  [ 6400/60000]
loss: 0.855333  [12800/60000]
loss: 1.009532  [19200/60000]
loss: 0.896136  [25600/60000]
loss: 0.920882  [32000/60000]
loss: 0.977126  [38400/60000]
loss: 0.914969  [44800/60000]
loss: 0.945779  [51200/60000]
loss: 0.897760  [57600/60000]
Epoch 3
-------------------------------
loss: 0.955482  [    0/60000]
loss: 0.990753  [ 6400/60000]
loss: 0.779999  [12800/60000]
loss: 0.947006  [19200/60000]
loss: 0.843262  [25600/60000]
loss: 0.856487  [32000/60000]
loss: 0.926935  [38400/60000]
loss: 0.868720  [44800/60000]
loss: 0.891358  [51200/60000]
loss: 0.85

In [13]:
# save the model
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


In [14]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

model.eval()
x, y = test_dataloder.dataset[0][0], test_dataloder.dataset[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')


Predicted: "Ankle boot", Actual: "Ankle boot"
